In [ ]:
!python -m pip install xgboost scikit-learn

In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
import random
import warnings
warnings.filterwarnings('ignore')

print("🚀 BOOTING HYPER-CALIBRATED, MULTI-FEATURE PREDICTIVE ENGINE 🚀")

# ==============================================================================
# 1. ADAPTIVE HISTORICAL ELO ENGINE (CONCEPT 1: RECENCY VOLATILITY)
# ==============================================================================
print("\n[1/5] Processing historical match records with adaptive K-factors...")
url = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"
matches = pd.read_csv(url)
matches['date'] = pd.to_datetime(matches['date'])
modern_matches = matches[matches['date'] >= '2006-01-01'].copy()

modern_matches['outcome'] = np.where(
    modern_matches['home_score'] > modern_matches['away_score'], 2,
    np.where(modern_matches['home_score'] == modern_matches['away_score'], 1, 0)
)

max_date = modern_matches['date'].max()
home_arr = modern_matches['home_team'].to_numpy()
away_arr = modern_matches['away_team'].to_numpy()
outcome_arr = modern_matches['outcome'].to_numpy()
tournament_arr = modern_matches['tournament'].to_numpy()
date_arr = modern_matches['date'].to_numpy()

home_elos = np.zeros(len(modern_matches))
away_elos = np.zeros(len(modern_matches))
elo_ratings = {}

for i in range(len(modern_matches)):
    home, away, tourney = home_arr[i], away_arr[i], tournament_arr[i]
    if home not in elo_ratings: elo_ratings[home] = 1500
    if away not in elo_ratings: elo_ratings[away] = 1500
    
    home_elos[i] = elo_ratings[home]
    away_elos[i] = elo_ratings[away]
    
    actual = 1.0 if outcome_arr[i] == 2 else 0.5 if outcome_arr[i] == 1 else 0.0
    expected = 1 / (1 + 10 ** ((elo_ratings[away] - elo_ratings[home]) / 400))
    
    # Base tournament stake weighting
    base_k = 50 if tourney == 'FIFA World Cup' else 30 if any(x in tourney for x in ['Qualifiers', 'Euro', 'Copa']) else 10
    
    # FEATURE 1: Time-decay scale factor to heavily penalize ancient eras
    years_back = (max_date - pd.Timestamp(date_arr[i])).days / 365.25
    recency_scale = np.exp(-0.12 * years_back) # Ancient matches approach zero structural impact
    adaptive_k = base_k * recency_scale
    
    elo_shift = adaptive_k * (actual - expected)
    elo_ratings[home] += elo_shift
    elo_ratings[away] -= elo_shift

modern_matches['home_elo'] = home_elos
modern_matches['away_elo'] = away_elos
modern_matches['elo_difference'] = modern_matches['home_elo'] - modern_matches['away_elo']

# Encode Geographics
sa_teams = {'Brazil', 'Argentina', 'Uruguay', 'Colombia', 'Chile', 'Peru', 'Ecuador', 'Paraguay'}
eu_teams = {'Germany', 'France', 'Spain', 'Italy', 'England', 'Portugal', 'Netherlands', 'Croatia', 'Belgium', 'Switzerland'}
def assign_continent(team): return 'SA' if team in sa_teams else 'EU' if team in eu_teams else 'Other'

modern_matches['home_continent'] = modern_matches['home_team'].apply(assign_continent)
modern_matches['away_continent'] = modern_matches['away_team'].apply(assign_continent)
le = LabelEncoder()
modern_matches['home_continent_encoded'] = le.fit_transform(modern_matches['home_continent'])
modern_matches['away_continent_encoded'] = le.transform(modern_matches['away_continent'])

# Train Historical Core
features = ['home_elo', 'away_elo', 'elo_difference', 'home_continent_encoded', 'away_continent_encoded']
X = modern_matches[features]
y = modern_matches['outcome'].astype(int)
days_back = (modern_matches['date'].max() - modern_matches['date']).dt.days
sample_weights = np.exp(-0.0006 * days_back)

xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X, y, sample_weight=sample_weights)

# ==============================================================================
# 2. SQUAD INGESTION (CONCEPTS 2, 3, & 4: TROPHIES, UNIFORMITY, VALUE)
# ==============================================================================
print("[2/5] Parsing player registries to calculate structural alignment metrics...")
fifa_df = pd.read_csv('ea_fc26_players.csv', low_memory=False)
nat_col, rating_col = 'nationality', 'overallRating'

# Exponential value curve for valuation mapping
fifa_df['value_proxy'] = np.where(fifa_df[rating_col] >= 70, ((fifa_df[rating_col] - 60) ** 3.6) * 110000, 450000)

# FEATURE 2: Real-world Modern Trophy Weights (Silverware Buff)
trophy_weights = {
    'Argentina': 0.05,  # World Cup 2022 & Copa América 2024
    'Spain': 0.05,      # Euro 2024 Winners
    'France': 0.02,     # World Cup 2022 Finalists
    'Italy': 0.01       # Euro 2020 Legacy
}

def aggregate_national_stats(group):
    top_11 = group.nlargest(11, rating_col)[rating_col]
    top_23 = group.nlargest(23, rating_col)
    
    # FEATURE 3: Squad Uniformity Index (Standard Deviation of Top 11)
    # Lower standard deviation indicates high balance/no critical weak links
    uniformity_std = top_11.std() if len(top_11) >= 11 else 5.0
    
    return pd.Series({
        'rating_mean': top_11.mean(),
        'rating_std': uniformity_std,
        'value_m': top_23['value_proxy'].sum() / 1000000
    })

nt_df = fifa_df.groupby(nat_col).apply(aggregate_national_stats).reset_index()
nt_df[nat_col] = nt_df[nat_col].replace({
    'United States': 'USA', 'Republic of Ireland': 'Ireland', 'Korea Republic': 'South Korea', 
    'Czech Republic': 'Czechia', 'China PR': 'China', 'Bosnia & Herzegovina': 'Bosnia and Herzegovina', 'IR Iran': 'Iran'
})

# ==============================================================================
# 3. ADVANCED COMBINATORIAL INFERENCE PIPELINE
# ==============================================================================
def run_match_inference(team1, team2, is_knockout=False):
    # Retrieve Squad Attributes
    r1 = nt_df.loc[nt_df[nat_col] == team1, 'rating_mean'].values[0] if team1 in nt_df[nat_col].values else 70.0
    r2 = nt_df.loc[nt_df[nat_col] == team2, 'rating_mean'].values[0] if team2 in nt_df[nat_col].values else 70.0
    
    std1 = nt_df.loc[nt_df[nat_col] == team1, 'rating_std'].values[0] if team1 in nt_df[nat_col].values else 4.0
    std2 = nt_df.loc[nt_df[nat_col] == team2, 'rating_std'].values[0] if team2 in nt_df[nat_col].values else 4.0
    
    v1 = nt_df.loc[nt_df[nat_col] == team1, 'value_m'].values[0] if team1 in nt_df[nat_col].values else 5.0
    v2 = nt_df.loc[nt_df[nat_col] == team2, 'value_m'].values[0] if team2 in nt_df[nat_col].values else 5.0

    # 1. Core Talent Foundation
    talent_baseline = 0.5 + ((r1 - r2) * 0.035)
    
    # 2. FEATURE 3: Cohesion / Uniformity Scaling
    # Teams with higher internal consistency (lower std dev) out-pass volatile star-reliant teams
    uniformity_modifier = (std2 - std1) * 0.015
    
    # 3. FEATURE 4: Sigmoid Premium Value Elasticity
    # Amplified exponential boost once squad depth value crosses the €800M threshold
    sigmoid_v1 = 1 / (1 + np.exp(-0.004 * (v1 - 800)))
    sigmoid_v2 = 1 / (1 + np.exp(-0.004 * (v2 - 800)))
    value_elasticity = (sigmoid_v1 - sigmoid_v2) * 0.12
    
    # 4. FEATURE 2: Modern Trophy Buffs
    trophy_modifier = trophy_weights.get(team1, 0.0) - trophy_weights.get(team2, 0.0)
    
    # 5. Background Historical Pedigree (Dampened to 15% to act purely as a tiebreaker)
    elo1, elo2 = elo_ratings.get(team1, 1500), elo_ratings.get(team2, 1500)
    c1, c2 = le.transform([assign_continent(team1)])[0], le.transform([assign_continent(team2)])[0]
    xgb_preds = xgb_model.predict_proba(pd.DataFrame([[elo1, elo2, elo1 - elo2, c1, c2]], columns=features))[0]
    pedigree_modifier = ((xgb_preds[2] + (xgb_preds[1] * 0.5)) - 0.5) * 0.15
    
    # Comprehensive Blend Layer
    blended_probability = np.clip(talent_baseline + uniformity_modifier + value_elasticity + trophy_modifier + pedigree_modifier, 0.02, 0.98)
    
    if is_knockout:
        # Knockout pressure curves
        if blended_probability > 0.54: blended_probability = min(0.99, blended_probability + 0.12)
        elif blended_probability < 0.46: blended_probability = max(0.01, blended_probability - 0.12)
        winner = team1 if np.random.rand() < blended_probability else team2
        return winner, blended_probability if winner == team1 else (1 - blended_probability)
    else:
        draw_margin = max(0.08, 0.24 - abs(blended_probability - 0.5) * 0.22)
        win_margin = blended_probability * (1 - draw_margin)
        dice_roll = np.random.rand()
        if dice_roll < win_margin: return 3, 0
        elif dice_roll < win_margin + draw_margin: return 1, 1
        return 0, 3

# ==============================================================================
# 4. BRACKET MAPPING & CHRONOLOGICAL SIMULATION
# ==============================================================================
print("[4/5] Constructing formal structural scheduling templates...")

world_cup_2026_groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czechia'], 'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'], 'D': ['USA', 'Paraguay', 'Australia', 'Turkey'], 
    'E': ['Germany', 'Curaçao', 'Ivory Coast', 'Ecuador'], 'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'], 'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'], 'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'], 'L': ['England', 'Croatia', 'Ghana', 'Panama']
}

st_data = {"group_stage": {}, "wildcards": [], "bracket_rounds": {}}
group_points = {g: {t: 0 for t in teams} for g, teams in world_cup_2026_groups.items()}

for g, teams in world_cup_2026_groups.items():
    fixtures = [(teams[0], teams[1]), (teams[2], teams[3]), (teams[0], teams[2]), (teams[1], teams[3]), (teams[0], teams[3]), (teams[1], teams[2])]
    for t1, t2 in fixtures:
        pts1, pts2 = run_match_inference(t1, t2)
        group_points[g][t1] += pts1
        group_points[g][t2] += pts2

w, ru, thirds = {}, {}, []
for g, teams in world_cup_2026_groups.items():
    ranked = sorted(group_points[g].items(), key=lambda x: (x[1], elo_ratings.get(x[0], 1500)), reverse=True)
    w[g], ru[g] = ranked[0][0], ranked[1][0]
    thirds.append({"team": ranked[2][0], "points": ranked[2][1]})

thirds.sort(key=lambda x: (x['points'], elo_ratings.get(x['team'], 1500)), reverse=True)
wc = [t['team'] for t in thirds[:8]]

# OFFICIAL FIFA WORLD CUP 2026 ANNEX C COMBINATORIAL SCHEDULING TREE
knockout_grid = [
    (ru['A'], ru['B']),         (w['E'], wc[0]), 
    (w['F'], ru['C']),          (w['C'], ru['F']),
    (w['I'], wc[1]),            (ru['E'], ru['I']),
    (w['A'], wc[2]),            (w['L'], wc[3]),
    (w['D'], wc[4]),            (w['G'], wc[5]),
    (ru['K'], ru['L']),         (w['H'], ru['J']),
    (w['B'], wc[6]),            (w['J'], ru['H']),
    (w['K'], wc[7]),            (ru['D'], ru['G'])
]

stages = ["Round of 32", "Round of 16", "Quarter-Finals", "Semi-Finals", "World Cup Final"]
current_tier = knockout_grid

print("[5/5] Executing high-fidelity structural tournament simulations...")
for stage_name in stages:
    st_data["bracket_rounds"][stage_name] = []
    next_tier_teams = []
    print(f"\n🏆 {stage_name.upper()} 🏆")
    
    for side_a, side_b in current_tier:
        victor, margin_prob = run_match_inference(side_a, side_b, is_knockout=True)
        print(f"   ⚔️ {side_a} vs {side_b} ➔ {victor} advances ({margin_prob*100:.1f}%)")
        next_tier_teams.append(victor)
        
    current_tier = [(next_tier_teams[j], next_tier_teams[j+1]) for j in range(0, len(next_tier_teams), 2)]

print(f"\n🌟🌟🌟 ULTIMATE 2026 CHAMPION: {current_tier[0][0].upper()} 🌟🌟🌟")

🚀 BOOTING HYPER-CALIBRATED, MULTI-FEATURE PREDICTIVE ENGINE 🚀

[1/5] Processing historical match records with adaptive K-factors...
[2/5] Parsing player registries to calculate structural alignment metrics...
[4/5] Constructing formal structural scheduling templates...
[5/5] Executing high-fidelity structural tournament simulations...

🏆 ROUND OF 32 🏆
   ⚔️ South Korea vs Bosnia and Herzegovina ➔ South Korea advances (67.9%)
   ⚔️ Ecuador vs France ➔ France advances (99.0%)
   ⚔️ Sweden vs Scotland ➔ Sweden advances (70.4%)
   ⚔️ Morocco vs Tunisia ➔ Morocco advances (88.9%)
   ⚔️ Senegal vs Brazil ➔ Brazil advances (86.6%)
   ⚔️ Ivory Coast vs Norway ➔ Norway advances (99.0%)
   ⚔️ South Africa vs Colombia ➔ Colombia advances (99.0%)
   ⚔️ England vs Japan ➔ England advances (82.8%)
   ⚔️ Turkey vs Austria ➔ Turkey advances (67.8%)
   ⚔️ Belgium vs Iran ➔ Belgium advances (99.0%)
   ⚔️ Uzbekistan vs Ghana ➔ Ghana advances (86.5%)
   ⚔️ Spain vs Algeria ➔ Spain advances (94.3%)
   ⚔️ S

IndexError: list index out of range